In [0]:
from pyspark.sql.functions import current_timestamp

# ==========================================
# 02 - GENERIC BRONZE LOAD VALIDATION
# ==========================================

configs = (
    spark.table("mortgage_catalog.metadata.ingestion_config")
    .filter("is_active = true")
    .collect()
)

failed_tables = []

for config in configs:

    target_schema = config["target_schema"]
    target_table  = config["target_table"]

    table_name = (
        f"mortgage_catalog."
        f"{target_schema}."
        f"{target_table}"
    )

    print(f"\nChecking → {table_name}")

    # Check table exists
    if not spark.catalog.tableExists(table_name):

        print(f"FAILED → Table not found: {table_name}")
        failed_tables.append(table_name)
        continue

    # Read Bronze table
    df = spark.table(table_name)

    row_count = df.count()

    print(f"Row Count : {row_count}")

    # Check required ingestion metadata
    required_columns = [
        "_ingestion_timestamp",
        "_source_system",
        "_source_type"
    ]

    missing_columns = [
        c for c in required_columns
        if c not in df.columns
    ]

    if missing_columns:

        print(
            f"FAILED → Missing columns: "
            f"{missing_columns}"
        )

        failed_tables.append(table_name)

    else:

        print(f"BRONZE VALIDATION SUCCESS → {table_name}")


# ==========================================
# FINAL JOB STATUS
# ==========================================

if failed_tables:

    raise Exception(
        f"Bronze validation failed: {failed_tables}"
    )

print("\nALL BRONZE TABLES VALIDATED ✅")


Checking → mortgage_catalog.bronze.customer_bronze
Row Count : 9
BRONZE VALIDATION SUCCESS → mortgage_catalog.bronze.customer_bronze

Checking → mortgage_catalog.bronze.property_bronze
Row Count : 3
BRONZE VALIDATION SUCCESS → mortgage_catalog.bronze.property_bronze

Checking → mortgage_catalog.bronze.branch_bronze
Row Count : 3
BRONZE VALIDATION SUCCESS → mortgage_catalog.bronze.branch_bronze

Checking → mortgage_catalog.bronze.document_bronze
Row Count : 3
BRONZE VALIDATION SUCCESS → mortgage_catalog.bronze.document_bronze

Checking → mortgage_catalog.bronze.sql_customer_bronze
Row Count : 5
BRONZE VALIDATION SUCCESS → mortgage_catalog.bronze.sql_customer_bronze

Checking → mortgage_catalog.bronze.loan_application_bronze
Row Count : 5
BRONZE VALIDATION SUCCESS → mortgage_catalog.bronze.loan_application_bronze

Checking → mortgage_catalog.bronze.payment_bronze
Row Count : 5
BRONZE VALIDATION SUCCESS → mortgage_catalog.bronze.payment_bronze

Checking → mortgage_catalog.bronze.credit_s